# 4 — Visualization & Insight

Tahap visualisasi: menampilkan hubungan antar variabel (contoh: domino effect NPL → TWP90) dan tren prediksi vs aktual.

**Input:**
- `3_modelling/output/3_model_predictions.csv`
- `2_data_preprocessing/output/2.2_final_feature_set.csv`

**Output (folder):** `4_visualization/output/` (gambar + `final_report.csv`)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / '3_modelling' / 'output' / '3_model_predictions.csv').exists():
            return p
    raise FileNotFoundError('Could not find 3_model_predictions.csv in current or parent directories.')

ROOT = find_project_root(Path.cwd())
pred_path = ROOT / '3_modelling' / 'output' / '3_model_predictions.csv'
features_path = ROOT / '2_data_preprocessing' / 'output' / '2.2_final_feature_set.csv'
metrics_path = ROOT / '3_modelling' / 'output' / 'model_metrics.csv'
output_dir = ROOT / '4_visualization' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

df_pred = pd.read_csv(pred_path)
df_pred['tanggal'] = pd.to_datetime(df_pred['tanggal'])
df_feat = pd.read_csv(features_path)
df_feat['tanggal'] = pd.to_datetime(df_feat['tanggal'])

print(f'Loaded predictions: {pred_path} | rows={len(df_pred):,}')
print(f'Loaded features: {features_path} | rows={len(df_feat):,}')

if metrics_path.exists():
    display(pd.read_csv(metrics_path))

In [ ]:
# Visual Proof: The Domino Effect (NPL shifted +6 months vs TWP90)
prov_id_demo = 1
data_demo = df_feat[df_feat['provinsi_id'] == prov_id_demo].sort_values('tanggal').copy()

if data_demo.empty:
    raise ValueError(f'No rows found for provinsi_id={prov_id_demo} in feature set.')

fig, ax1 = plt.subplots(figsize=(14, 7))

color_twp = 'tab:red'
ax1.set_xlabel('Date')
ax1.set_ylabel('TWP90 (%)', color=color_twp, fontweight='bold')
ax1.plot(data_demo['tanggal'], data_demo['twp90_pct'], color=color_twp, linewidth=3, label='TWP90 (Actual)')
ax1.tick_params(axis='y', labelcolor=color_twp)

ax2 = ax1.twinx()
color_npl = 'tab:blue'
ax2.set_ylabel('NPL Ratio (Shifted +6m)', color=color_npl, fontweight='bold')
data_demo['tanggal_shifted'] = data_demo['tanggal'] + pd.DateOffset(months=6)
ax2.plot(
    data_demo['tanggal_shifted'],
    data_demo['x9_npl_ratio'],
    color=color_npl,
    linestyle='--',
    linewidth=2,
    alpha=0.7,
    label='NPL (Leading Indicator - 6m ago)',
)
ax2.tick_params(axis='y', labelcolor=color_npl)

plt.title(f'The Domino Effect: Bank NPL Leads P2P Defaults (Province {prov_id_demo})', fontsize=14)

lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc='upper left')
plt.grid(True, alpha=0.3)
fig.tight_layout()

out_domino = output_dir / f'domino_effect_prov_{prov_id_demo}.png'
fig.savefig(out_domino, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_domino}')

In [ ]:
# Grafik tren prediksi vs aktual (berdasarkan file output modelling)
prov_id_demo = 1
pred_demo = df_pred[df_pred['provinsi_id'] == prov_id_demo].sort_values('tanggal').copy()

if pred_demo.empty:
    print(f'No prediction rows found for provinsi_id={prov_id_demo}.')
else:
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(pred_demo['tanggal'], pred_demo['y_true'], label='Actual', linewidth=2)
    ax.plot(pred_demo['tanggal'], pred_demo['y_pred'], label='Predicted', linewidth=2, linestyle='--')
    ax.set_title(f'Prediction vs Actual (Province {prov_id_demo})')
    ax.set_xlabel('Date')
    ax.set_ylabel('TWP90 (%)')
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    out_pred = output_dir / f'pred_vs_actual_prov_{prov_id_demo}.png'
    fig.savefig(out_pred, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_pred}')

In [ ]:
# Simpan laporan akhir berbasis prediksi (row-level)
final_report = df_pred.copy()

if 'y_true' in final_report.columns and 'y_pred' in final_report.columns:
    final_report['abs_error'] = (final_report['y_pred'] - final_report['y_true']).abs()

# Tambahkan x9_npl_ratio jika tersedia (join ke feature set)
join_cols = ['provinsi_id', 'tanggal']
if all(c in df_feat.columns for c in join_cols) and 'x9_npl_ratio' in df_feat.columns:
    final_report = final_report.merge(
        df_feat[join_cols + ['x9_npl_ratio']],
        on=join_cols,
        how='left',
    )

assert (final_report['provinsi_id'] == 19).sum() == 0, 'provinsi_id 19 still present (expected removed upstream).'

report_path = output_dir / 'final_report.csv'
final_report.to_csv(report_path, index=False)
print(f'Saved: {report_path}')